# DataSense Latency Profiler

Loads `sanjaymalladi/DataSense-Modal-E2B-SFT` (Gemma-4-E2B-it + LoRA) and runs the same **THINK → EXECUTE → DEBUG → ANSWER** agent loop as the Titanic/F1 eval notebook, but instruments every stage with timers so we can see exactly *where* the latency lives before deciding whether MTP / speculative decoding / cuDF acceleration is worth building.

**Stages timed, per question:**
1. `model_load` — one-time, timed separately in Section 1
2. `tokenize` — building the chat-template input
3. `generate` — `model.generate()` wall time, per turn
4. `sandbox_exec` — E2B `run_code()` wall time, per turn (this includes network round-trip to E2B's cloud, not just compute)
5. `parse` — regex extraction of code/answer from model text, per turn

Output: a per-question, per-turn timing table plus a stacked breakdown chart so you can see the % of total latency spent in LLM generation vs sandbox execution vs network/parsing overhead. That tells you whether it's worth optimizing token generation (MTP/speculative decoding) or execution (cuDF) or both.

## 0. Setup — install dependencies

In [ ]:
%%capture
!pip install -q -U unsloth
!pip install -q -U e2b-code-interpreter
!pip install -q -U bitsandbytes accelerate peft transformers
!pip install -q -U matplotlib seaborn pandas

In [ ]:
import os

# --- Secrets ---
# Kaggle: Add-ons -> Secrets -> add E2B_API_KEY (get one free at https://e2b.dev)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    E2B_API_KEY = user_secrets.get_secret("E2B_API_KEY")
except Exception:
    E2B_API_KEY = os.environ.get("E2B_API_KEY", "")

if not E2B_API_KEY:
    raise ValueError(
        "E2B_API_KEY not found. Add it via Kaggle Secrets (Add-ons > Secrets) "
        "or set os.environ['E2B_API_KEY'] manually in this cell."
    )

os.environ["E2B_API_KEY"] = E2B_API_KEY
print("E2B key loaded:", E2B_API_KEY[:6] + "..." if E2B_API_KEY else "MISSING")

## 1. Load DataSense — timed

In [ ]:
import time

timing_log = []  # every stage timing across the whole run goes here

def log_time(question_id, turn, stage, seconds, extra=None):
    timing_log.append({
        "question_id": question_id,
        "turn": turn,
        "stage": stage,
        "seconds": seconds,
        "extra": extra or "",
    })

_t0 = time.perf_counter()

from unsloth import FastLanguageModel
import torch

BASE_MODEL = "unsloth/gemma-4-E2B-it"
LORA_ADAPTER = "sanjaymalladi/DataSense-Modal-E2B-SFT"
MAX_SEQ_LENGTH = 4096

_t_load_start = time.perf_counter()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
_t_base_loaded = time.perf_counter()

model.load_adapter(LORA_ADAPTER)
_t_adapter_loaded = time.perf_counter()

FastLanguageModel.for_inference(model)
_t_ready = time.perf_counter()

log_time("_startup", 0, "base_model_load", _t_base_loaded - _t_load_start)
log_time("_startup", 0, "lora_adapter_load", _t_adapter_loaded - _t_base_loaded)
log_time("_startup", 0, "inference_mode_switch", _t_ready - _t_adapter_loaded)
log_time("_startup", 0, "total_model_load", _t_ready - _t_load_start)

print(f"Base model load:   {_t_base_loaded - _t_load_start:.2f}s")
print(f"LoRA adapter load: {_t_adapter_loaded - _t_base_loaded:.2f}s")
print(f"Inference switch:  {_t_ready - _t_adapter_loaded:.2f}s")
print(f"TOTAL model load:  {_t_ready - _t_load_start:.2f}s")
print("Loaded:", BASE_MODEL, "+", LORA_ADAPTER)

## 2. E2B Sandbox executor (timed)

In [ ]:
from e2b_code_interpreter import Sandbox

class E2BExecutor:
    """Same as the eval notebook's executor, but every .run() call records
    its own wall-clock time (round-trip to the E2B cloud sandbox, including
    network latency -- this is NOT local compute time)."""

    def __init__(self, timeout=1200):
        self.timeout = timeout
        _t = time.perf_counter()
        self.sbx = Sandbox.create(timeout=timeout)
        self.sandbox_create_seconds = time.perf_counter() - _t
        self._uploaded = []

    def upload_csv(self, local_path: str, remote_name: str) -> str:
        remote_path = f"/home/user/{remote_name}"
        _t = time.perf_counter()
        with open(local_path, "rb") as f:
            self.sbx.files.write(remote_path, f)
        self.upload_seconds = time.perf_counter() - _t
        self._uploaded.append((local_path, remote_name))
        return remote_path

    def _recreate(self):
        print("[sandbox expired -- recreating and re-uploading files]")
        self.sbx = Sandbox.create(timeout=self.timeout)
        for local_path, remote_name in self._uploaded:
            remote_path = f"/home/user/{remote_name}"
            with open(local_path, "rb") as f:
                self.sbx.files.write(remote_path, f)

    def run(self, code: str, _retried=False):
        """Executes code, returns dict with stdout, stderr, error, ok flag,
        and elapsed_seconds for this specific call."""
        _t = time.perf_counter()
        try:
            self.sbx.set_timeout(self.timeout)
            execution = self.sbx.run_code(code)
        except Exception as e:
            if not _retried and ("not found" in str(e).lower() or "timeout" in str(e).lower()):
                self._recreate()
                return self.run(code, _retried=True)
            return {"ok": False, "stdout": "", "stderr": "", "result_text": "", "error": str(e),
                    "elapsed_seconds": time.perf_counter() - _t}

        elapsed = time.perf_counter() - _t
        stdout = "\n".join(log for log in execution.logs.stdout) if execution.logs.stdout else ""
        stderr = "\n".join(log for log in execution.logs.stderr) if execution.logs.stderr else ""
        result_text = execution.text or ""
        error = None
        if execution.error:
            error = f"{execution.error.name}: {execution.error.value}"
        ok = error is None
        return {
            "ok": ok,
            "stdout": stdout,
            "stderr": stderr,
            "result_text": result_text,
            "error": error,
            "elapsed_seconds": elapsed,
        }

    def close(self):
        try:
            self.sbx.kill()
        except Exception:
            pass

print("E2BExecutor (instrumented) ready.")

## 3. Agent loop helpers (prompt building, response parsing)

In [ ]:
import re

SYSTEM_PROMPT = """You are DataSense, a personal data-science agent. You answer questions about \
tabular datasets by writing and executing real Python code -- never by guessing.

CRITICAL RULES ABOUT VARIABLES AND COLUMNS:
1. The dataset(s) are ALREADY LOADED in the sandbox. Use EXACTLY the variable name(s) given in
   "Dataset schema" below. NEVER invent a variable name like `db`, `data`, or `dataset` -- if the
   schema says the variable is `dfs`, you must write `dfs["tablename"]`, nothing else.
2. ONLY use column names that are EXPLICITLY listed in the schema below. Do not guess or assume a
   column exists (e.g. do not assume `position == 1` means a win unless told so -- check the sample
   rows and exact column names given).
3. Look at the sample rows in the schema carefully -- string/categorical columns may use different
   exact values than you'd expect (e.g. "1" vs 1 vs "1st"). Match the real values you are shown.

RESPONSE RULES:
4. Respond with a single ```python code block that computes the answer and prints it clearly.
5. Do not say **Answer:** until you have seen real execution output for that code.
6. Once you have real output, on the FINAL turn respond with:
   **Answer:** <value>
   **Summary:** <one sentence>
7. If your code errored, do not repeat the exact same code -- diagnose the traceback and change
   your approach (different column, different variable, different merge key, etc.).
"""

CODE_BLOCK_RE = re.compile(r"```(?:python)?\s*\n(.*?)```", re.DOTALL)
ANSWER_RE = re.compile(r"\*\*Answer:\*\*\s*(.+)")
SUMMARY_RE = re.compile(r"\*\*Summary:\*\*\s*(.+)")


def extract_code(text):
    matches = CODE_BLOCK_RE.findall(text)
    return matches[-1].strip() if matches else None


def extract_answer(text):
    m = ANSWER_RE.search(text)
    return m.group(1).strip() if m else None


def extract_summary(text):
    m = SUMMARY_RE.search(text)
    return m.group(1).strip() if m else None


def generate_timed(messages, max_new_tokens=512):
    """Returns (decoded_text, tokenize_seconds, generate_seconds, num_new_tokens)."""
    _t = time.perf_counter()
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    tokenize_seconds = time.perf_counter() - _t

    _t = time.perf_counter()
    out = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generate_seconds = time.perf_counter() - _t

    new_tokens = out[0][inputs.shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return text, tokenize_seconds, generate_seconds, len(new_tokens)


print("Agent helpers (instrumented) ready.")

## 4. The agent loop, instrumented at every stage

In [ ]:
def run_agent_timed(question_id: str, question: str, schema_info: str, executor: "E2BExecutor",
                     max_turns: int = 4, verbose: bool = True):
    """Same THINK -> EXECUTE -> DEBUG -> ANSWER loop as the eval notebook, but every
    stage (tokenize, generate, sandbox exec, parse) is timed and appended to the
    global `timing_log` list, tagged with question_id and turn number."""
    messages = [
        {"role": "user", "content": (
            f"{SYSTEM_PROMPT}\n\nDataset schema:\n{schema_info}\n\n"
            f"Question: {question}"
        )}
    ]

    exec_results = []
    any_exec_ok = False
    final_text = ""
    prev_code_block = None
    _question_t0 = time.perf_counter()

    for turn in range(max_turns):
        model_text, tok_s, gen_s, n_tokens = generate_timed(messages)
        log_time(question_id, turn + 1, "tokenize", tok_s)
        log_time(question_id, turn + 1, "generate", gen_s, extra=f"{n_tokens} new tokens")
        messages.append({"role": "assistant", "content": model_text})
        final_text = model_text

        _t = time.perf_counter()
        code_block = extract_code(model_text)
        answer = extract_answer(model_text)
        log_time(question_id, turn + 1, "parse", time.perf_counter() - _t)

        if verbose:
            print(f"--- {question_id} | Turn {turn+1} | generate={gen_s:.2f}s tokens={n_tokens} ---")

        if code_block:
            result = executor.run(code_block)
            log_time(question_id, turn + 1, "sandbox_exec", result["elapsed_seconds"],
                      extra="ok" if result["ok"] else "error")
            exec_results.append(result)
            any_exec_ok = any_exec_ok or result["ok"]

            if verbose:
                status = "OK" if result["ok"] else "ERROR"
                print(f"[exec {status}] sandbox_exec={result['elapsed_seconds']:.2f}s")

            if result["ok"] and answer:
                break

            is_repeat = (not result["ok"] and prev_code_block is not None
                         and code_block.strip() == prev_code_block.strip())
            prev_code_block = code_block

            if is_repeat:
                nudge = ("You submitted the EXACT SAME code that just failed. Do not repeat it. "
                          "Re-read the traceback and the schema's column list, pick a DIFFERENT "
                          "column or approach.")
            else:
                nudge = ("This succeeded. If you're confident, give your final "
                          "**Answer:** and **Summary:** now."
                          if result["ok"] else
                          "This failed. Fix the code and try again.")
            feedback = (
                f"Execution result:\nstdout: {result['stdout']}\n"
                f"stderr: {result['stderr']}\n"
                f"error: {result['error']}\n\n" + nudge
            )
            messages.append({"role": "user", "content": feedback})
        else:
            if answer:
                break
            messages.append({"role": "user", "content":
                "You must write executable Python code in a ```python block before answering."})

    total_seconds = time.perf_counter() - _question_t0
    log_time(question_id, 0, "question_total", total_seconds)

    return {
        "question_id": question_id,
        "question": question,
        "final_text": final_text,
        "exec_results": exec_results,
        "exec_ok": any_exec_ok,
        "predicted_answer": extract_answer(final_text),
        "turns_used": len(exec_results) if exec_results else 1,
        "total_seconds": total_seconds,
    }

print("Instrumented agent loop ready.")

## 5. Load a dataset (Titanic) and upload to E2B

Add the Titanic dataset via **Add Input** in the right sidebar before running (search "Titanic" → official competition dataset). Swap in F1 or your own CSV the same way if you want a second data-shape comparison.

In [ ]:
import pandas as pd, glob

titanic_candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True) \
    + glob.glob("/kaggle/input/**/titanic*.csv", recursive=True) \
    + glob.glob("/kaggle/input/**/Titanic*.csv", recursive=True)
titanic_candidates = [p for p in titanic_candidates if "test" not in p.lower()]

if not titanic_candidates:
    raise FileNotFoundError(
        "No Titanic CSV found under /kaggle/input. Add the Titanic dataset via "
        "'Add Input' in the sidebar."
    )

TITANIC_PATH = titanic_candidates[0]
titanic_df = pd.read_csv(TITANIC_PATH)
print("Using:", TITANIC_PATH, titanic_df.shape)

executor = E2BExecutor()
log_time("_startup", 0, "sandbox_create", executor.sandbox_create_seconds)

remote_path = executor.upload_csv(TITANIC_PATH, "titanic.csv")
log_time("_startup", 0, "csv_upload", executor.upload_seconds)

setup_code = f'''
import pandas as pd
df = pd.read_csv("{remote_path}")
print(df.shape)
'''
setup_result = executor.run(setup_code)
assert setup_result["ok"], setup_result["error"]
print(f"Sandbox created in {executor.sandbox_create_seconds:.2f}s, "
      f"CSV uploaded in {executor.upload_seconds:.2f}s")

schema_info = (
    "The dataframe is ALREADY LOADED in the sandbox as the variable `df`.\n"
    f"Columns and dtypes:\n{titanic_df.dtypes.to_string()}\n\n"
    f"Shape: {titanic_df.shape}\n\n"
    f"Sample rows:\n{titanic_df.head(3).to_string()}"
)

## 6. Run a handful of questions through the timed agent loop

Mix of easy (single-turn) and harder (multi-turn/retry-prone) questions so the timing table reflects realistic variance, not just the best case.

In [ ]:
QUESTIONS = [
    ("q1_survival_rate", "What percentage of passengers survived?"),
    ("q2_avg_fare_by_class", "What is the average fare for each passenger class?"),
    ("q3_survival_by_sex", "What was the survival rate broken down by sex?"),
    ("q4_oldest_survivor", "What is the age of the oldest passenger who survived?"),
    ("q5_embarked_counts", "How many passengers embarked from each port?"),
]

results = []
for qid, q in QUESTIONS:
    print(f"\n{'='*60}\n{qid}: {q}\n{'='*60}")
    r = run_agent_timed(qid, q, schema_info, executor, max_turns=4, verbose=True)
    results.append(r)
    print(f"-> total={r['total_seconds']:.2f}s | turns={r['turns_used']} | exec_ok={r['exec_ok']} | answer={r['predicted_answer']}")

## 7. Timing breakdown — where does the latency actually go?

In [ ]:
timing_df = pd.DataFrame(timing_log)
timing_df.to_csv("timing_log.csv", index=False)

print("Raw timing log (first 20 rows):")
display(timing_df.head(20))

# Per-stage totals across the whole run (excludes _startup one-off costs)
per_stage = (
    timing_df[~timing_df["question_id"].eq("_startup") & ~timing_df["stage"].eq("question_total")]
    .groupby("stage")["seconds"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
)
per_stage["pct_of_total"] = (per_stage["sum"] / per_stage["sum"].sum() * 100).round(1)
print("\nPer-stage totals (across all questions/turns):")
display(per_stage)

print("\nOne-time startup costs:")
display(timing_df[timing_df["question_id"] == "_startup"][["stage", "seconds"]])

print("\nPer-question totals:")
display(timing_df[timing_df["stage"] == "question_total"][["question_id", "seconds"]])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overall share of time by stage (pie)
axes[0].pie(per_stage["sum"], labels=per_stage.index, autopct="%1.1f%%", startangle=90)
axes[0].set_title("Share of total latency by stage\n(generate / sandbox_exec / tokenize / parse)")

# Right: stacked bar per question, showing generate vs sandbox_exec vs other
pivot = (
    timing_df[~timing_df["question_id"].eq("_startup") & ~timing_df["stage"].eq("question_total")]
    .groupby(["question_id", "stage"])["seconds"].sum().unstack(fill_value=0)
)
pivot = pivot.reindex(columns=[c for c in ["tokenize", "generate", "sandbox_exec", "parse"] if c in pivot.columns])
pivot.plot(kind="bar", stacked=True, ax=axes[1], colormap="viridis")
axes[1].set_ylabel("seconds")
axes[1].set_title("Per-question latency breakdown")
axes[1].legend(title="stage")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.savefig("latency_breakdown.png", dpi=150)
plt.show()

## 8. Takeaway

Read `per_stage` above:
- If **`generate`** dominates → the bottleneck is token-by-token LLM decoding. This is where MTP / speculative decoding would actually help.
- If **`sandbox_exec`** dominates → the bottleneck is E2B round-trip / pandas execution, not the model. This is where the `cudf.pandas` injection plan pays off, not token-generation tricks.
- If they're roughly balanced → both optimizations matter, and the cuDF path is still the lower-effort win since it needs zero retraining.

`timing_log.csv` and `latency_breakdown.png` are saved to the working directory — pull both into the pitch deck as your acceleration-proof evidence.

In [ ]:
executor.close()
print("Sandbox closed.")